# Stock Market Prediction using LSTM

A time-series forecasting project that uses an LSTM neural network to predict stock closing prices from historical market data.

## Technologies
- Python
- NumPy
- Pandas
- Matplotlib
- Scikit-learn
- TensorFlow / Keras

## Workflow
1. Load historical stock data
2. Select and normalize closing prices
3. Create time-series sequences
4. Train an LSTM model
5. Predict prices on test data
6. Calculate RMSE
7. Visualize actual vs predicted prices


In [ ]:
# Install the required packages if running in a new environment
# !pip install yfinance numpy pandas matplotlib scikit-learn tensorflow


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
    from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout


## 1. Load Historical Stock Data

The example below uses Google (`GOOGL`) historical data. The ticker can be changed if required.

In [ ]:
ticker = "GOOGL"
data = yf.download(ticker, start="2015-01-01", end="2025-01-01", auto_adjust=True)

data.head()


In [ ]:
close_data = data[["Close"]].copy()
    close_data.dropna(inplace=True)
close_data.describe()


## 2. Normalize the Data

Min-Max scaling converts the closing prices to a range between 0 and 1, which helps the neural network train more effectively.

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(close_data)

scaled_data[:5]


## 3. Create Time-Series Sequences

The model uses the previous 60 trading days to predict the next closing price.

In [ ]:
sequence_length = 60
X, y = [], []

for i in range(sequence_length, len(scaled_data)):
    X.append(scaled_data[i-sequence_length:i, 0])
    y.append(scaled_data[i, 0])

X = np.array(X)
y = np.array(y)

# Reshape for LSTM: samples, time steps, features
X = X.reshape((X.shape[0], X.shape[1], 1))

print("X shape:", X.shape)
print("y shape:", y.shape)


## 4. Train-Test Split

The data is split chronologically so that future observations are not used to train the model.

In [ ]:
split_index = int(len(X) * 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## 5. Build the LSTM Model

In [ ]:
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1)
])

model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()


## 6. Train the Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


## 7. Generate Predictions


In [ ]:
predictions_scaled = model.predict(X_test)

predictions = scaler.inverse_transform(predictions_scaled)
actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))


## 8. Evaluate using RMSE


In [ ]:
rmse = np.sqrt(mean_squared_error(actual_prices, predictions))
print(f"Test RMSE: ${rmse:.2f}")


## 9. Actual vs Predicted Closing Prices

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(actual_prices, label="Actual Closing Price")
plt.plot(predictions, label="Predicted Closing Price")
plt.title(f"{ticker} Stock Price: Actual vs Predicted")
plt.xlabel("Test Time Steps")
plt.ylabel("Closing Price")
plt.legend()
plt.show()


## Result

The LSTM model learns patterns from historical closing prices and produces predictions for the test period. RMSE is used as the evaluation metric, where a lower value indicates smaller prediction errors on the test data.

### Note
This project is intended for machine-learning practice and does not constitute financial advice or a reliable method for predicting future market movements.